# AI in Healthcare — RAG with LangChain & FAISS

## Practical Challenge

This project is a modified version of the LangChain + FAISS RAG demo provided during the Agentic AI Bootcamp.

As part of the practical challenge, the original demo was adapted to a new domain: **AI in Healthcare**.

### Our Modifications

* Replaced the original YOLO-NAS and DeciCoder sources with **5 healthcare-related websites**.
* Added document loading and processing for all five sources.
* Used OpenAI embeddings and FAISS for vector storage and similarity search.
* Modified the retriever to return the **Top 6 most relevant document chunks**.
* Designed **3 different user personas**, each with a different question.
* Used RAG to generate answers based on the retrieved information from the collected sources.

### User Scenarios

1. **Hospital Manager** — asks about the benefits of AI in hospitals and patient care.
2. **Medical Researcher** — asks about clinical applications of AI in diagnosis and screening.
3. **Regulatory/Compliance Officer** — asks about regulation, safety, and risks of AI-enabled medical devices.

### Technologies

* Python
* LangChain
* FAISS
* OpenAI Embeddings
* Retrieval-Augmented Generation (RAG)
* Google Colab



### 0. Install dependencies

In [2]:
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install sentence-transformers
!pip install faiss-cpu
!pip install tiktoken
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.5 MB/s eta 0:00:00


### 1. Set up API Key

In [3]:
import os
import openai
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## 1. Index System — Load the 5 websites about AI in Healthcare

We picked 5 websites/articles covering AI in Healthcare from different angles: a health-org overview, a hospital-system perspective, a clinical research review, a tech-company perspective, and a regulatory (FDA) perspective.

## Sources

1. **WHO — Health Topics: Artificial Intelligence**
   Global health policy overview.

2. **Mayo Clinic Press — AI in Healthcare**
   Hospital and patient care perspective.

3. **PMC — AI in Healthcare: Narrative Review of Clinical Applications**
   Academic and clinical research perspective.

4. **IBM Think — AI in Healthcare**
   Industry and technology perspective.

5. **Wikipedia —  Artificial intelligence in healthcare**



In [4]:
from langchain_community.document_loaders import WebBaseLoader

# 5 websites related to "AI in Healthcare"
who_loader = WebBaseLoader("https://www.who.int/health-topics/artificial-intelligence").load()
mayo_loader = WebBaseLoader("https://mcpress.mayoclinic.org/healthy-aging/ai-in-healthcare-the-future-of-patient-care-and-health-management/").load()
pmc_review_loader = WebBaseLoader("https://pmc.ncbi.nlm.nih.gov/articles/PMC12764347/").load()
ibm_loader = WebBaseLoader("https://www.ibm.com/think/topics/artificial-intelligence-healthcare").load()
wiki_loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare").load()

/tmp/ipykernel_1141/580754233.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [5]:
# Quick check that each source loaded correctly
for name, loader in [
    ("WHO", who_loader),
    ("Mayo Clinic", mayo_loader),
    ("PMC Review", pmc_review_loader),
    ("IBM", ibm_loader),
    ("Wikipedia", wiki_loader),
]:
    print(f"{name}: {len(loader)} document loaded")

WHO: 1 document loaded
Mayo Clinic: 1 document loaded
PMC Review: 1 document loaded
IBM: 1 document loaded
Wikipedia: 1 document loaded


### Chunk documents

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len
)

# Use the split_documents method
who_chunks = text_splitter.split_documents(who_loader)
mayo_chunks = text_splitter.split_documents(mayo_loader)
pmc_review_chunks = text_splitter.split_documents(pmc_review_loader)
ibm_chunks = text_splitter.split_documents(ibm_loader)
wiki_chunks = text_splitter.split_documents(wiki_loader)

In [12]:

print(f"Number of chunks - WHO: {len(who_chunks)}")
print(f"Number of chunks - Mayo Clinic: {len(mayo_chunks)}")
print(f"Number of chunks - PMC Review: {len(pmc_review_chunks)}")
print(f"Number of chunks - IBM: {len(ibm_chunks)}")
print(f"Number of chunks - Wikipedia: {len(wiki_chunks)}")

Number of chunks - WHO: 8
Number of chunks - Mayo Clinic: 1
Number of chunks - PMC Review: 202
Number of chunks - IBM: 67
Number of chunks - Wikipedia: 318


### Create the index (embeddings + FAISS vector store, with caching)

In [13]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.storage import LocalFileStore
import time
import os

# 1. Set up the local cache store
store_path = "./cache/"
if not os.path.exists(store_path):
    os.makedirs(store_path)
    print(f"Created cache directory: {store_path}")

store = LocalFileStore(store_path)
print(f"Cache store initialized at {store_path}")
print("-" * 30)

# 2. Create the core embeddings model
print("Initializing the core OpenAI embedding model (text-embedding-3-small)...")
core_embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Create the cache-backed embedder
print("Initializing the CacheBackedEmbeddings wrapper...")
embedder = CacheBackedEmbeddings.from_bytes_store(
    core_embeddings_model,
    store,
    namespace=core_embeddings_model.model
)
print("Embedder is now configured to use the cache.")
print("-" * 30)

Created cache directory: ./cache/
Cache store initialized at ./cache/
------------------------------
Initializing the core OpenAI embedding model (text-embedding-3-small)...
Initializing the CacheBackedEmbeddings wrapper...
Embedder is now configured to use the cache.
------------------------------


/usr/local/lib/python3.13/dist-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [34]:
# 4. Create the FAISS vector store from the first source, then add the rest

#1
print(f"Adding {len(who_chunks)} documents from 'WHO' to the FAISS vector store.")
start_time = time.time()
vectorstore = FAISS.from_documents(who_chunks, embedder)
end_time = time.time()
print(f"FAISS vector store created. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains {vectorstore.index.ntotal} vectors.")
print("-" * 30)

#2
print(f"Adding {len(mayo_chunks)} documents from 'Mayo Clinic' to the FAISS vector store.")
start_time = time.time()
vectorstore.add_documents(mayo_loader)
end_time = time.time()
print(f"FAISS vector store created. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains {vectorstore.index.ntotal} vectors.")
print("-" * 30)

#3
print(f"Adding {len(pmc_review_chunks)} documents from 'pmc review' to the FAISS vector store.")
start_time = time.time()
vectorstore.add_documents(pmc_review_chunks)
end_time = time.time()
print(f"FAISS vector store created. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains {vectorstore.index.ntotal} vectors.")
print("-" * 30)

#4
print(f"Adding {len(ibm_chunks)} documents from 'IBM' to the FAISS vector store.")
start_time = time.time()
vectorstore.add_documents(ibm_chunks)
end_time = time.time()
print(f"FAISS vector store created. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains {vectorstore.index.ntotal} vectors.")
print("-" * 30)

#5
print(f"Adding {len(wiki_chunks)} documents from 'Wikipedia' to the FAISS vector store.")
start_time = time.time()
vectorstore.add_documents(wiki_chunks)
end_time = time.time()
print(f"FAISS vector store created. Time taken: {end_time - start_time:.2f} seconds.")
print(f"The vector store now contains {vectorstore.index.ntotal} vectors.")
print("-" * 30)


Adding 8 documents from 'WHO' to the FAISS vector store.
FAISS vector store created. Time taken: 0.01 seconds.
The vector store now contains 8 vectors.
------------------------------
Adding 1 documents from 'Mayo Clinic' to the FAISS vector store.
FAISS vector store created. Time taken: 0.00 seconds.
The vector store now contains 9 vectors.
------------------------------
Adding 202 documents from 'pmc review' to the FAISS vector store.
FAISS vector store created. Time taken: 0.12 seconds.
The vector store now contains 211 vectors.
------------------------------
Adding 67 documents from 'IBM' to the FAISS vector store.
FAISS vector store created. Time taken: 0.04 seconds.
The vector store now contains 278 vectors.
------------------------------
Adding 318 documents from 'Wikipedia' to the FAISS vector store.
FAISS vector store created. Time taken: 0.19 seconds.
The vector store now contains 596 vectors.
------------------------------


### Instantiate the retriever — return **Top 6** documents

This is the key change requested: instead of the default `k` (usually 4), we configure the retriever to return the **top 6** most relevant chunks (`search_kwargs={"k": 6}`).


In [35]:
# 7. Instantiate a retriever configured to return the TOP 6 most relevant document
print("Converting the vector store into a retriever with Top 6 documents.")
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
print("Retriever is ready!")
print("-" * 30)


# Let's see if the cache works!
print("Re-adding the same documents to demonstrate caching...")
start_time = time.time()
vectorstore.add_documents(who_chunks) # This should be very fast!
end_time = time.time()
print(f"Time taken to re-add cached documents: {end_time - start_time:.4f} seconds.")
print("-" * 30)

print("Process complete. You can now use the 'retriever' object for similarity searches.")

Converting the vector store into a retriever with Top 6 documents.
Retriever is ready!
------------------------------
Re-adding the same documents to demonstrate caching...
Time taken to re-add cached documents: 0.0102 seconds.
------------------------------
Process complete. You can now use the 'retriever' object for similarity searches.


## 2. Retrieval System & 3. Augment System

We now wire the retriever (Top 6) into a `RetrievalQA` chain with an LLM, so retrieved context automatically augments the prompt sent to the model.


In [36]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_classic.callbacks import StdOutCallbackHandler

handler = StdOutCallbackHandler()

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0,
    streaming=False,
    callbacks=[handler],
)

# This is the entire retrieval + augment system (retriever returns top 6 chunks)
qa_with_sources_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    callbacks=[handler],
    return_source_documents=True
)

## Demo: 3 different users asking 3 different questions

Each "user" represents a different persona interested in AI in Healthcare, asking a different type of question. The system retrieves the top 6 relevant chunks from our 5 sources and answers using that context.


### User 1 — A hospital administrator asking about cost/efficiency

In [37]:
response = qa_with_sources_chain({"query":"How can AI help hospitals reduce costs and improve operational efficiency?"})
#only response
print(response['result'])
#source
print(response['source_documents'])



> Entering new RetrievalQA chain...

> Finished chain.
AI can help hospitals reduce costs and improve operational efficiency in several ways:

1. **Billing and Fraud Detection:** AI systems like IBM’s DataProbe analyze billing records to identify false claims and billing errors, recovering significant amounts of money and reducing waste.

2. **Predictive Analytics:** AI models analyze electronic health records (EHRs) to predict patients at high risk of readmission or complications, enabling targeted preventive care that can reduce costly hospital stays.

3. **Supply and Staffing Management:** AI predicts supply needs and helps manage staffing levels efficiently, ensuring resources are used optimally without overstocking or understaffing.

4. **Reducing Duplicate Tests:** By analyzing patient data, AI can help avoid unnecessary duplicate tests, saving costs and reducing patient burden.

5. **Workload Management:** AI automates administrative tasks, prioritizes patient needs, and facil

### User 2 — A medical researcher, asking about clinical applications

In [38]:
response = qa_with_sources_chain({"query":"What are some clinical applications of AI in healthcare?"})
#only response
print(response['result'])
#source
print(response['source_documents'])



> Entering new RetrievalQA chain...

> Finished chain.
Some clinical applications of AI in healthcare include:

1. Disease Diagnosis: AI is used to support decision making and predictive modeling in primary care, helping physicians choose appropriate treatments. It has also been applied in cancer diagnosis, including reading imaging studies and pathology, risk stratification, molecular characterization of tumors, and predicting optimal treatment protocols based on individual patient characteristics.

2. Workload Management: AI algorithms can automate administrative tasks, prioritize patient needs, and facilitate communication within healthcare teams, thereby streamlining care coordination and reducing workload.

3. Imaging Diagnostics: AI improves accuracy and efficiency in interpreting medical images, aiding in early and precise diagnosis.

4. Patient Monitoring: AI systems can continuously monitor patient data to detect changes in condition and alert healthcare providers promptly.


### User 3 — A compliance officer, asking about oversight and safety

In [39]:
response = qa_with_sources_chain({"query":"How are AI systems in healthcare regulated, and what safety measures are used to reduce potential risks ?"})
#only response
print(response['result'])
#source
print(response['source_documents'])



> Entering new RetrievalQA chain...

> Finished chain.
AI systems in healthcare are regulated through a combination of privacy rules, ethical guidelines, and oversight by regulatory bodies to ensure patient safety and data protection. For example, in the United States, the Office for Civil Rights (OCR) has issued rules requiring healthcare providers to protect individuals' health information privacy when using AI. Providers must keep records of how AI is used and ensure the security of AI systems.

Safety measures to reduce potential risks include:

- Ensuring patient informed consent by informing patients when AI is involved in their care and explaining its role.

- Addressing the "black box" nature of many AI systems through efforts in "explainable AI," which aims to provide understandable rationales for AI recommendations (e.g., highlighting features that led to a prediction).

- Monitoring and mitigating algorithmic bias to prevent reproduction of existing health inequalities.

-

## Summary

- We collected content from **5 websites** about **AI in Healthcare** (WHO, Mayo Clinic, a PMC clinical review, IBM, and the Wikipedia).
- We chunked and embedded the content into a **FAISS** vector store with a **cache-backed embedder**.
- We configured the retriever to return the **Top 6** documents (`k=6`) per query, instead of the default.
- We simulated **3 different users** (hospital administrator, medical researcher, regulatory officer), each asking a different question, and the RAG system answered each using the retrieved context plus listed the sources used.
